In [1]:
%load_ext cuml.accel

cuML: Installed accelerator for sklearn.
cuML: Successfully initialized accelerator.


In [2]:

import os
import argparse
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression, Lasso, LinearRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (
    mean_squared_error,
    accuracy_score,
    roc_auc_score,
    balanced_accuracy_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

from cache.cache_utils import load_saved_data
from src.utils import *

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
TRANSFORM_TARGETS = True
NORMALIZE_FEATURES = True
NR_LAYERS = 32
DATASET_NAME = "mmlu_high_school"
MODEL_NAME = "swiss-ai/Apertus-8B-Instruct-2509"
SAVE_NAME = ""
SEED = 52
SAVE_CACHE_KEY = "3000"
SAVE_DIR = "/users/astepancic/projects/apertus-probes/scratch/mera-runs"
NR_ATTEMPTS = 5 # for CV
MAX_TRIALS = 5 # for non-convergence retries
ERROR_TYPE = "SM" # SM or CE (squared error or cross-entropy)
EPSILON = 1e-10 # for counting non-zero coefficients



In [4]:
METRICS ={"classification": {"AUCROC": roc_auc_score, "ACC": accuracy_score, "BACC": balanced_accuracy_score},\
              "regression": {"MSE": mean_squared_error} }

probe_results = []

# FIXME: there is no SEED in cuml models
def initialise_regression_models(seed: int, alphas) -> dict:
    """Initialise regression models with various hyperparameters."""
    models = {
        f"L-{alpha}": Ridge(
            alpha=alpha, fit_intercept=False, max_iter=5000
        )
        for alpha in alphas
    }  # positive=True,
    models["L-0"] = LinearRegression(fit_intercept=False)
    return models



alphas = [5, 2, 1, 0.5, 0.25, 0.1]
logistic = LogisticRegression(max_iter=5000, C=0.75, penalty='l1', fit_intercept=False)
MODELS = {
    "classification": {"LogReg-l1": logistic},
    "regression": initialise_regression_models(SEED, alphas),
}

Unused keyword parameter: max_iter during cuML estimator initialization
Unused keyword parameter: max_iter during cuML estimator initialization
Unused keyword parameter: max_iter during cuML estimator initialization
Unused keyword parameter: max_iter during cuML estimator initialization
Unused keyword parameter: max_iter during cuML estimator initialization
Unused keyword parameter: max_iter during cuML estimator initialization


In [5]:
results_path = f'{SAVE_DIR}/{DATASET_NAME}/{MODEL_NAME.split("/")[1]}/{SAVE_CACHE_KEY}_results.pkl'
os.makedirs(os.path.dirname(results_path), exist_ok=True)


activations_path = f'{SAVE_DIR}/{DATASET_NAME}/{MODEL_NAME.split("/")[1]}/{SAVE_CACHE_KEY}_acts.pkl'
print(activations_path)
with open(activations_path, 'rb') as f:
    activations = pickle.load(f)
        
targets_path = f'{SAVE_DIR}/{DATASET_NAME}/{MODEL_NAME.split("/")[1]}/{SAVE_CACHE_KEY}_targets.pkl'
print("Targets path:", targets_path)
with open(targets_path, 'rb') as f:
    targets = pickle.load(f)

SAVE_RESULTS_PATH = f'{SAVE_DIR}/{DATASET_NAME}/{MODEL_NAME.split("/")[1]}/{SAVE_CACHE_KEY}_probe_results'
print(f"Saving results to {SAVE_RESULTS_PATH}")



/users/astepancic/projects/apertus-probes/scratch/mera-runs/mmlu_high_school/Apertus-8B-Instruct-2509/3000_acts.pkl
Targets path: /users/astepancic/projects/apertus-probes/scratch/mera-runs/mmlu_high_school/Apertus-8B-Instruct-2509/3000_targets.pkl
Saving results to /users/astepancic/projects/apertus-probes/scratch/mera-runs/mmlu_high_school/Apertus-8B-Instruct-2509/3000_probe_results


The `activations` variable is a dictionary that contains the following keys:
- `activations_cache`: A dictionary where each key is an integer (layer index) and the value is a NumPy array representing the activations for that layer.
- `y_correct`: A list of boolean values indicating whether the predictions were correct for each sample.
- `y_error_sm`: A NumPy array representing the softmax error for each sample.
- `y_error_ce`: A list of cross-entropy errors for each sample.
- `activations_cache_exact`: Similar to `activations_cache`, but for exact matches.
- `y_correct_exact`: A list of boolean values indicating whether the predictions were correct for exact matches.
- `y_error_sm_exact`: A NumPy array representing the softmax error for exact matches.
- `y_error_ce_exact`: A list of cross-entropy errors for exact matches.

### Example: Accessing the activations for layer 0


In [6]:
activations_path = f'{SAVE_DIR}/{DATASET_NAME}/{MODEL_NAME.split("/")[1]}/{SAVE_CACHE_KEY}_acts.pkl'
print(activations_path)
print(os.path.exists(os.path.dirname(activations_path)))
print(os.path.exists(activations_path))

/users/astepancic/projects/apertus-probes/scratch/mera-runs/mmlu_high_school/Apertus-8B-Instruct-2509/3000_acts.pkl
True
True


In [7]:
exact_activations = activations["activations_cache_exact"]
y_error_exact = activations["y_error_sm_exact"] if ERROR_TYPE == "SM" else activations["y_error_ce_exact"]
y_correct_exact = activations["y_correct_exact"]
print(exact_activations[0][0].shape)



(4096,)


In [8]:
y_error_exact

array([0.33224595, 0.62914455, 0.26111948, ..., 0.65405136, 0.5721271 ,
       0.95145449])

In [10]:
for task_type, model_dict in MODELS.items():
    print("Task type:", task_type)
    for model_name, model in model_dict.items():
        print("Model:", model_name)
        for layer_idx, layer_data in tqdm(exact_activations.items(), desc="Processing Features"):
            print("Layer", layer_idx)
            print(layer_data.shape)
            assert isinstance(layer_data, np.ndarray)
            X = layer_data
            if NORMALIZE_FEATURES:
                print("normalizeing features")
                X = StandardScaler().fit_transform(X)
                print("Feature means (first 5):", X.mean(axis=0)[:5])
                print("Feature stds (first 5):", X.std(axis=0)[:5])
            if task_type == "classification":
                y_true = y_correct_exact
            elif task_type == "regression":
                y_true = y_error_exact
                if TRANSFORM_TARGETS and task_type == "regression":
                    y_true = np.clip(y_true, 1e-8, 1 - 1e-8)
                    y_true = np.log(y_true / (1 - y_true))
            else:
                raise ValueError(f"Unknown task type: {task_type}")
            # Transform to log-odd-ratio space
          

            # FIXME Why use normalise error?
            #   elif normalise_error and model_task == "regression":
            #         y_true /= y_true.max()

         
            for m in range(NR_ATTEMPTS):
                print("Layer:", layer_idx, "Attempt:", m)
                
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y_true, test_size=0.3, random_state=SEED + m
                )
                # print number of parameters in X_train and number of samples

                dummy_model = (
                    DummyClassifier(strategy="most_frequent")
                    if task_type == "classification"
                    else DummyRegressor(strategy="mean")
                ) 
                dummy_model.fit(X_train, y_train)
                dummy_y_pred = dummy_model.predict(X_test)
                dummy_metrics = {
                    f"Dummy-{metric_name}": metric(
                        y_test, dummy_y_pred
                    )
                    for metric_name, metric in METRICS[task_type].items()
                }
                no_coeffs = True
                trials = 0
                while no_coeffs and trials < MAX_TRIALS:
                    trials += 1
                    model.fit(X_train, y_train)
                    coef_norm = np.linalg.norm(model.coef_)
                    s = np.linalg.svd(X_train, compute_uv=False)
                    quantile = np.quantile(s, [0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0])
                    cond_est = s[0] / s[-1] if s[-1] > 0 else np.inf
                    print("coef_norm:", coef_norm, "cond_est:", cond_est, "quantiles:", quantile)
                    # print("||w||:", coef_norm)  # track vs. layer; exploding norms → near-separable/hard
                    # print("n_iter_:", getattr(model, "n_iter_", None))  # per class for OvR
                    # print(f"cond_est≈{cond_est:.1e}, y_balance={np.mean(y_train):.3f}") # track vs. layer; high cond_est → near-separable/hard

                    y_pred = model.predict(X_test)

                    # Collect coefficients and count non-zero ones.
                    assert hasattr(model, "coef_") # using sklearn's LogisticRegression
                    non_zero_coeffs = np.where(np.abs(model.coef_) > EPSILON)[0]
                    no_coeffs = len(non_zero_coeffs) == 0
                if no_coeffs:
                    print(f"Warning: No non-zero coefficients found after {MAX_TRIALS} trials at layer {layer_idx}, attempt {m}.")
                
                results = {
                    **{
                        metric_name: metric(y_test, y_pred)
                        for metric_name, metric in METRICS[task_type].items()
                    },
                    **dummy_metrics,
                }
                print("Results:", results)

                if not no_coeffs:
                    result_line = {
                        "Dataset": DATASET_NAME,
                        "LLM_Model": MODEL_NAME,
                        "Task": task_type,
                        "Probe Model": model_name,
                        "Layer": layer_idx,
                        "Coefficients": model.coef_.flatten(),
                        "Coef-norm": coef_norm,
                        "Cond-Est": cond_est,
                        "# of non-zero coefficients": len(non_zero_coeffs),
                        "Attempt": m,
                        "Token-Pos": "exact",
                        "y_pred": y_pred,
                        "y_test": y_test,
                        **results,
                    }
                    probe_results.append(result_line)
df_results = pd.DataFrame(probe_results)
            

Task type: classification
Model: LogReg-l1


Processing Features:   0%|          | 0/32 [00:00<?, ?it/s]

Layer 0
(3000, 4096)
normalizeing features
Feature means (first 5): [ 3.5941600e-08  1.3887882e-07 -5.7091317e-08  7.5538956e-08
 -1.7520676e-07]
Feature stds (first 5): [1.0000057 1.0000015 1.0000007 1.0000051 0.9999935]
Layer: 0 Attempt: 0
coef_norm: 0.26063547 cond_est: inf quantiles: [nan nan nan nan nan nan nan nan nan]
Results: {'AUCROC': 0.5764033056531459, 'ACC': 0.5733333333333334, 'BACC': 0.5764033056531459, 'Dummy-AUCROC': 0.5, 'Dummy-ACC': 0.5511111111111111, 'Dummy-BACC': 0.5}
Layer: 0 Attempt: 1
coef_norm: 0.20941232 cond_est: inf quantiles: [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 3.83087765e-28 1.05847111e+02
 1.54751599e+03]
Results: {'AUCROC': 0.5818037558875255, 'ACC': 0.5811111111111111, 'BACC': 0.5818037558875255, 'Dummy-AUCROC': 0.5, 'Dummy-ACC': 0.5755555555555556, 'Dummy-BACC': 0.5}
Layer: 0 Attempt: 2
coef_norm: 0.18006283 cond_est: inf quantiles: [nan nan nan nan nan nan nan nan nan]
Results: {'AUCROC': 0.5914

Processing Features:   3%|▎         | 1/32 [00:06<03:15,  6.29s/it]

coef_norm: 0.18739344 cond_est: inf quantiles: [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 1.69609858e-28 1.03344335e+02
 1.55694043e+03]
Results: {'AUCROC': 0.5658696341162799, 'ACC': 0.5244444444444445, 'BACC': 0.5658696341162799, 'Dummy-AUCROC': 0.5, 'Dummy-ACC': 0.5677777777777778, 'Dummy-BACC': 0.5}
Layer 1
(3000, 4096)
normalizeing features
Feature means (first 5): [ 1.2318293e-08  4.3312709e-09 -2.3206075e-08  3.5007794e-08
  1.0112921e-08]
Feature stds (first 5): [1.0000002  0.99999976 0.9999994  1.         1.0000004 ]
Layer: 1 Attempt: 0
coef_norm: 5.001931 cond_est: 2.0415797e+20 quantiles: [7.44505813e-18 2.91733967e-15 4.62008398e-02 1.37910545e-01
 4.07266960e-01 1.30523586e+00 9.32610793e+00 1.53900246e+02
 1.51996790e+03]
Results: {'AUCROC': 0.5938597892047269, 'ACC': 0.5933333333333334, 'BACC': 0.5938597892047269, 'Dummy-AUCROC': 0.5, 'Dummy-ACC': 0.5511111111111111, 'Dummy-BACC': 0.5}
Layer: 1 Attempt: 1
coef_norm: 5.0885

Processing Features:   3%|▎         | 1/32 [00:13<07:02, 13.62s/it]


KeyboardInterrupt: 

In [ ]:
df_results.to_pickle(SAVE_RESULTS_PATH + '.pkl')

NameError: name 'df_results' is not defined

In [ ]:
# I want to create essential results and save it to csv.
essential_results = df_results[[
    "Dataset", "LLM_Model", "Task", "Probe Model", "Error-Type", "Layer",
    "# of non-zero coefficients", "Attempt", "Token-Pos"
] + list(METRICS.keys())]
essential_results.to_csv(SAVE_RESULTS_PATH + '.csv', index=False)
essential_results.head()

,Dataset,LLM_Model,Task,Probe Model,Error-Type,Layer,# of non-zero coefficients,Attempt,Token-Pos,AUCROC,ACC,BACC
0,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,0,exact,0.570515,0.605556,0.570515
1,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,1,exact,0.555762,0.608889,0.555762
2,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,2,exact,0.561055,0.596667,0.561055
3,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,3,exact,0.572098,0.615556,0.572098
4,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,4,exact,0.561118,0.610000,0.561118


In [ ]:
completions_path = f'{SAVE_DIR}/{DATASET_NAME}/{MODEL_NAME.split("/")[1]}/{SAVE_CACHE_KEY}_completions.pkl'

with open(completions_path, "rb") as f:
    completions = pickle.load(f)
# print(completions.keys())
# print(completions["completions"][:2])
# print(completions["answers"][:2])
# Assuming `completions` contains a list of token sequences and `match_indices` contains indices
match_indices = completions["match_indices"]
generations = completions["completions"]  # Assuming `tokens` contains the token sequences
# Inspect the tokens at `match_indices` and surrounding tokens
for i, (generation, match_index) in enumerate(zip(generations, match_indices)):
    generation = generation.squeeze(0)
    if match_index < len(generation):  # Ensure the index is within bounds
        print(f"Sample {i}:")
        print("Generation:", len(generation))
        print("Match index: ", match_index)
        print(f"Matched Token: {generation[match_index]}")
        # Print tokens around the matched token (e.g., 5 tokens before and after)
        start = max(0, match_index - 5)
        end = min(len(tokens[i]), match_index + 6)
        print(f"Context: {' '.join(tokens[i][start:end])}")
        print("-" * 50)
    else:
        print(f"Sample {i}: Match index {match_index} is out of bounds for tokens.")


Sample 0:
Generation: 203
Match index:  105
Matched Token: 1349
Context: 
--------------------------------------------------
Sample 1:
Generation: 206
Match index:  106
Matched Token: 1407
Context: 
--------------------------------------------------
Sample 2:
Generation: 175
Match index:  75
Matched Token: 1359
Context: 
--------------------------------------------------
Sample 3:
Generation: 179
Match index:  79
Matched Token: 1407
Context: 
--------------------------------------------------
Sample 4:
Generation: 210
Match index:  110
Matched Token: 1349
Context: 
--------------------------------------------------
Sample 5:
Generation: 192
Match index:  92
Matched Token: 1349
Context: 
--------------------------------------------------
Sample 6:
Generation: 145
Match index:  73
Matched Token: 1398
Context: 
--------------------------------------------------
Sample 7:
Generation: 174
Match index:  74
Matched Token: 1398
Context: 
--------------------------------------------------
Sampl

TypeError: sequence item 0: expected str instance, numpy.ndarray found